# ICT324 · Final project · Traffy Fondue Insight
**Team Ctrl C + Ctrl V:** Aung Myo (6702955), Aye Chan Ko (6703266), Phoo Myat May Zaw (6703013)

**Research question:** Can data mining on Bangkok's Traffy Fondue complaints show **what** the problems are (from the text, even when the type is empty), **where** and **when** they cluster, and **why** some rise (rain)?

**What this notebook does (CRISP-DM, full pipeline)**
1. Loads the official Traffy Fondue CSV, describes it and cleans it, with a cleaning log
2. **What:** text labelling, majority baseline + Logistic Regression, Linear SVM, Random Forest, 10-fold stratified CV (seed 1992), mean ± std
3. Thai → English machine translation (NLLB-200), Thai vs. English test, English keywords
4. **Where:** DBSCAN hotspot map
5. **When / why:** seasonal index, hour × weekday, rain vs. flooding
6. **New for the final:** time-based test on 2025 · ROC curves and most confident mistakes · filling empty types with the model · Getis-Ord Gi* and HDBSCAN · association rules (FP-Growth) · priority score · next-week forecast (XGBoost) · data file for the monitoring app
7. Saves every table, chart and map into `final_outputs.zip`

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. About 60–90 minutes with full translation.
**Google Drive:** the first cell asks to connect your Google Drive. Allow it. Everything is saved in `MyDrive/ICT324_final_outputs`, so if Colab resets you lose nothing: run cell 1, then the Resume cell before section 13.
**Tip:** put last run's `translation_cache.csv` in that Drive folder to skip complaints already translated.


In [ ]:
# ---------- setup ----------
!pip -q install iterative-stratification==0.1.9 sentencepiece mlxtend xgboost

import os, re, time, json, warnings, zipfile
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

CSV_URL   = "https://publicapi.traffy.in.th/dump-csv-chadchart/bangkok_traffy.csv"
CSV_PATH  = "bangkok_traffy.csv"
OUT       = "final_outputs"         # everything we save goes here
SEED      = 1992                    # same seed as Lab 4
N_FOLDS   = 10                      # 10-fold stratified CV (Ajarn's rule)
SAMPLE_N  = 30000                   # complaints used for the text models (lower = faster)
TOP_K     = 10                      # number of problem types the text model predicts
TRAIN_START, TRAIN_END = "2022-06-01", "2025-10-31"   # labelled period
TEST_START = "2025-01-01"           # time-based test: train before this date, test after
FORCE_TYPES = ["น้ำท่วม"]          # always keep Flooding in the text model (it fell out of the top 10 after cleaning)
BKK_BOX   = dict(lat=(13.45, 14.05), lon=(100.30, 100.95))  # rough Bangkok bounding box
TRANSLATE = True                    # Thai -> English with NLLB-200 (needs Runtime > Change runtime type > T4 GPU)
MT_MODEL  = "facebook/nllb-200-distilled-600M"
MT_BATCH  = 32

SAVE_TO_DRIVE = True                # keep all outputs (and the data file) in Google Drive, so a Colab reset loses nothing
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")       # click "Connect to Google Drive" and allow
        OUT = "/content/drive/MyDrive/ICT324_final_outputs"
        CSV_PATH = OUT + "/bangkok_traffy.csv"
    except Exception as e:
        print("Google Drive not used:", e)
os.makedirs(OUT, exist_ok=True)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 80)

# Thai problem types -> English (for charts; matplotlib cannot draw Thai by default)
TYPE_EN = {"ถนน":"Road","ทางเท้า":"Sidewalk","ความปลอดภัย":"Safety","แสงสว่าง":"Street lighting",
           "กีดขวาง":"Obstruction","ความสะอาด":"Cleanliness","ร้องเรียน":"General complaint","จราจร":"Traffic",
           "ป้าย":"Signs","น้ำท่วม":"Flooding","ท่อระบายน้ำ":"Drainage","ต้นไม้":"Trees","คลอง":"Canal",
           "สะพาน":"Bridge","สายไฟ":"Power lines","เสียงรบกวน":"Noise","คนจรจัด":"Homeless people",
           "ห้องน้ำ":"Public toilets","ป้ายจราจร":"Traffic signs","PM2.5":"PM2.5","การเดินทาง":"Transport",
           "สอบถาม":"Inquiry","เสนอแนะ":"Suggestion"}
en = lambda t: TYPE_EN.get(t, t)
# 50 Bangkok districts -> English names (fixed list, so names are never mistranslated)
DISTRICT_EN = {"พระนคร":"Phra Nakhon","ดุสิต":"Dusit","หนองจอก":"Nong Chok","บางรัก":"Bang Rak","บางเขน":"Bang Khen",
 "บางกะปิ":"Bang Kapi","ปทุมวัน":"Pathum Wan","ป้อมปราบศัตรูพ่าย":"Pom Prap Sattru Phai","พระโขนง":"Phra Khanong",
 "มีนบุรี":"Min Buri","ลาดกระบัง":"Lat Krabang","ยานนาวา":"Yan Nawa","สัมพันธวงศ์":"Samphanthawong","พญาไท":"Phaya Thai",
 "ธนบุรี":"Thon Buri","บางกอกใหญ่":"Bangkok Yai","ห้วยขวาง":"Huai Khwang","คลองสาน":"Khlong San","ตลิ่งชัน":"Taling Chan",
 "บางกอกน้อย":"Bangkok Noi","บางขุนเทียน":"Bang Khun Thian","ภาษีเจริญ":"Phasi Charoen","หนองแขม":"Nong Khaem",
 "ราษฎร์บูรณะ":"Rat Burana","บางพลัด":"Bang Phlat","ดินแดง":"Din Daeng","บึงกุ่ม":"Bueng Kum","สาทร":"Sathon",
 "บางซื่อ":"Bang Sue","จตุจักร":"Chatuchak","บางคอแหลม":"Bang Kho Laem","ประเวศ":"Prawet","คลองเตย":"Khlong Toei",
 "สวนหลวง":"Suan Luang","จอมทอง":"Chom Thong","ดอนเมือง":"Don Mueang","ราชเทวี":"Ratchathewi","ลาดพร้าว":"Lat Phrao",
 "วัฒนา":"Watthana","บางแค":"Bang Khae","หลักสี่":"Lak Si","สายไหม":"Sai Mai","คันนายาว":"Khan Na Yao",
 "สะพานสูง":"Saphan Sung","วังทองหลาง":"Wang Thonglang","คลองสามวา":"Khlong Sam Wa","บางนา":"Bang Na",
 "ทวีวัฒนา":"Thawi Watthana","ทุ่งครุ":"Thung Khru","บางบอน":"Bang Bon"}
den = lambda d: DISTRICT_EN.get(str(d).strip(), d)
SEASON = {3:"Hot",4:"Hot",5:"Hot",6:"Rainy",7:"Rainy",8:"Rainy",9:"Rainy",10:"Rainy",11:"Cool",12:"Cool",1:"Cool",2:"Cool"}
print("Ready. Outputs go to:", OUT)

## 1. Load the data

In [ ]:
if not os.path.exists(CSV_PATH):
    !wget -q --show-progress -O {CSV_PATH} {CSV_URL}

COLS = ["ticket_id","type","organization","comment","coords","district","subdistrict",
        "timestamp","state","last_activity","count_reopen","photo"]
raw = pd.read_csv(CSV_PATH, usecols=lambda c: c in COLS, dtype=str, low_memory=False)
print(f"Rows: {len(raw):,}   Columns: {raw.shape[1]}")
raw.head(3)

## 2. Data understanding: missing values, duplicates, states

In [ ]:
desc = pd.DataFrame({
    "missing": raw.isna().sum(),
    "missing %": (raw.isna().mean()*100).round(2),
    "unique values": raw.nunique(),
})
desc.to_csv(f"{OUT}/01_missing_values.csv")
display(desc)

print("Duplicate ticket_id rows:", f"{raw['ticket_id'].duplicated().sum():,}")
print("\nState counts:"); print(raw["state"].value_counts(dropna=False).head(10))

## 3. Cleaning (with a log of what we changed)

In [ ]:
log = []
def step(df, what, why):
    log.append({"step": what, "rows after": len(df), "why": why}); return df

df = step(raw.copy(), "Start", "Raw download")
df = step(df.drop_duplicates("ticket_id"), "Drop duplicate tickets", "Same complaint counted twice")

# time: report time and last update, converted to Bangkok time
for c in ["timestamp","last_activity"]:
    df[c] = pd.to_datetime(df[c], errors="coerce", utc=True, format="mixed").dt.tz_convert("Asia/Bangkok")
df = step(df.dropna(subset=["timestamp"]), "Drop rows without a report time", "Cannot place them in time")
df = step(df[df["timestamp"] >= pd.Timestamp(TRAIN_START, tz="Asia/Bangkok")],
          "Drop early test records (before June 2022)", "System launched in mid-2022; earlier rows are tests")

# location: coords are 'lon,lat'
ll = df["coords"].str.split(",", expand=True)
df["lon"] = pd.to_numeric(ll[0], errors="coerce"); df["lat"] = pd.to_numeric(ll[1], errors="coerce")
inside = df["lat"].between(*BKK_BOX["lat"]) & df["lon"].between(*BKK_BOX["lon"])
df = step(df[inside], "Drop GPS points outside Bangkok", "Wrong or missing location")

# problem types: '{ถนน,ทางเท้า}' -> ['ถนน','ทางเท้า']
df["types"] = (df["type"].fillna("").str.strip("{}").str.split(",")
               .apply(lambda xs: [x.strip() for x in xs if x.strip()]))
df["labelled"] = df["types"].str.len() > 0
df["comment"] = df["comment"].fillna("").str.replace(r"http\S+", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()

# time features
df["year"], df["month"] = df["timestamp"].dt.year, df["timestamp"].dt.month
df["weekday"], df["hour"] = df["timestamp"].dt.dayofweek, df["timestamp"].dt.hour
df["season"] = df["month"].map(SEASON)
df["district_clean"] = df["district"].fillna("").str.strip().str.replace(r"^เขต\s*", "", regex=True)
df["district_en"] = df["district_clean"].map(DISTRICT_EN)            # NaN if not one of the 50 Bangkok districts
bad_d = df.loc[df["district_en"].isna(), "district"].value_counts()
log.append({"step": f"Check district names: {df['district_en'].notna().sum():,} rows match the 50 Bangkok districts",
            "rows after": len(df), "why": f"{len(bad_d)} other spellings/areas kept but excluded from district analysis"})
df["date"] = df["timestamp"].dt.date

# fixing time for finished cases + outlier flags
df["finished"] = df["state"].eq("เสร็จสิ้น")
df["fix_days"] = (df["last_activity"] - df["timestamp"]).dt.total_seconds() / 86400
df.loc[~df["finished"], "fix_days"] = np.nan
df["fix_outlier"] = df["fix_days"].lt(0) | df["fix_days"].gt(365)
df["count_reopen"] = pd.to_numeric(df["count_reopen"], errors="coerce").fillna(0)
df["has_photo"] = df["photo"].notna()

clean_log = pd.DataFrame(log); clean_log["rows removed"] = (-clean_log["rows after"].diff()).fillna(0).astype(int)
print("District names not matching the 50 Bangkok districts (top 10):"); print(bad_d.head(10))
clean_log.to_csv(f"{OUT}/02_cleaning_log.csv", index=False)
display(clean_log)
print(f"Clean rows: {len(df):,}   labelled: {df['labelled'].mean()*100:.1f}%   finished: {df['finished'].mean()*100:.1f}%")

## 4. Describe the clean data: outliers and coverage

In [ ]:
fd = df.loc[df["finished"] & ~df["fix_outlier"], "fix_days"]
q1, q3 = fd.quantile([.25,.75]); iqr = q3 - q1
outl = pd.DataFrame({
    "fix_days (finished, 0-365)": fd.describe().round(2),
}).T
outl["IQR outliers (> Q3+1.5·IQR)"] = int((fd > q3 + 1.5*iqr).sum())
outl["flagged <0 or >365 days"] = int(df["fix_outlier"].sum())
outl.to_csv(f"{OUT}/03_fix_days_outliers.csv"); display(outl)

empty_by_year = (~df["labelled"]).groupby(df["year"]).mean().mul(100).round(1).rename("empty type %")
empty_by_year.to_csv(f"{OUT}/04_empty_type_by_year.csv"); display(empty_by_year.to_frame())

## 5. What: label the problem type from the complaint text
- Labelled period June 2022 – October 2025, complaints with text, top `TOP_K` types (multi-label: one complaint can have several types)
- Text features: TF-IDF on **character n-grams (2–4)**, which works for Thai without a word tokenizer
- TF-IDF is fitted **inside each fold**, so the test fold never leaks into training
- **Same 10 stratified folds and seed 1992 for every model** (multi-label stratified K-fold)

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, hamming_loss

lab = df[df["labelled"] & (df["timestamp"] <= pd.Timestamp(TRAIN_END+" 23:59", tz="Asia/Bangkok")) & (df["comment"].str.len() >= 5)]
top = pd.Series([t for ts in lab["types"] for t in ts]).value_counts().head(TOP_K).index.tolist()
top += [t for t in FORCE_TYPES if t not in top]          # Flooding is needed for the rain analysis
lab = lab[lab["types"].apply(lambda ts: any(t in top for t in ts))]
n_before = len(lab)
lab = lab.assign(_norm=lab["comment"].str.lower().str.replace(r"\s+", "", regex=True)).drop_duplicates("_norm").drop(columns="_norm")
DUP_REMOVED = n_before - len(lab)
print(f"Removed {DUP_REMOVED:,} copy-pasted duplicate complaints ({DUP_REMOVED/n_before*100:.1f}%) so the same text cannot be in both train and test folds")
lab = lab.sample(min(SAMPLE_N, len(lab)), random_state=SEED)
X = lab["comment"].values
Y = np.array([[int(t in ts) for t in top] for ts in lab["types"]])
LABELS = [en(t) for t in top]
print(f"Text set: {len(lab):,} complaints  |  labels: {LABELS}")
print("Share of complaints with each label (%):", dict(zip(LABELS, (Y.mean(0)*100).round(1))))

In [ ]:
def tfidf(maxf): return TfidfVectorizer(analyzer="char_wb", ngram_range=(2,4), min_df=3, max_features=maxf, sublinear_tf=True)

MODELS = {   # ordered from simplest to most complex
  "Baseline (majority)": (Pipeline([("tfidf", tfidf(1000)), ("clf", OneVsRestClassifier(DummyClassifier(strategy="most_frequent")))]),
                          "predicts the most common value for every label"),
  "Logistic Regression": (Pipeline([("tfidf", tfidf(50000)), ("clf", OneVsRestClassifier(LogisticRegression(C=4, solver="liblinear", class_weight="balanced", max_iter=1000)))]),
                          "TF-IDF char 2-4, 50k features; C=4, liblinear, class_weight=balanced"),
  "Linear SVM":          (Pipeline([("tfidf", tfidf(50000)), ("clf", OneVsRestClassifier(LinearSVC(C=0.5, class_weight="balanced")))]),
                          "TF-IDF char 2-4, 50k features; C=0.5, class_weight=balanced"),
  "Random Forest":       (Pipeline([("tfidf", tfidf(5000)),  ("clf", RandomForestClassifier(n_estimators=100, min_samples_leaf=2, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED))]),
                          "TF-IDF char 2-4, 5k features; 100 trees, min_samples_leaf=2, balanced_subsample"),
}

def scores(model, Xt):
    clf = model.named_steps["clf"]
    if hasattr(clf, "decision_function") and not isinstance(clf, RandomForestClassifier):
        try: return model.decision_function(Xt)
        except Exception: pass
    p = model.predict_proba(Xt)
    if isinstance(p, list):   # Random Forest multi-label: one (n,2) array per label
        return np.column_stack([pi[:,1] if pi.shape[1] > 1 else np.zeros(len(Xt)) for pi in p])
    return p

OOF_SCORES = {}   # out-of-fold scores, used for ROC curves
def run_cv(models, Xs, Ys, folds, tag=""):
    rows, oof = [], {}
    for name, (model, params) in models.items():
        t0, fold_res, pred_all = time.time(), [], np.zeros_like(Ys)
        score_all = np.zeros(Ys.shape, dtype=float)
        for k, (tr, te) in enumerate(folds):
            model.fit(Xs[tr], Ys[tr])
            yp, ys = model.predict(Xs[te]), scores(model, Xs[te])
            pred_all[te] = yp; score_all[te] = ys
            try: auc = roc_auc_score(Ys[te], ys, average="macro")
            except ValueError: auc = np.nan
            fold_res.append(dict(
                macro_f1=f1_score(Ys[te], yp, average="macro", zero_division=0),
                micro_f1=f1_score(Ys[te], yp, average="micro", zero_division=0),
                precision=precision_score(Ys[te], yp, average="macro", zero_division=0),
                recall=recall_score(Ys[te], yp, average="macro", zero_division=0),
                auc=auc, hamming=hamming_loss(Ys[te], yp)))
        fr = pd.DataFrame(fold_res); oof[name] = pred_all; OOF_SCORES[tag+name] = score_all
        rows.append({"Model": name, **{f"{c} mean": fr[c].mean() for c in fr}, **{f"{c} std": fr[c].std() for c in fr},
                     "Parameters": params, "Time (s)": round(time.time()-t0)})
        print(f"{tag}{name:22s} macro F1 {fr.macro_f1.mean():.3f} ± {fr.macro_f1.std():.3f}   AUC {fr.auc.mean():.3f}   ({time.time()-t0:.0f}s)")
    return pd.DataFrame(rows), oof

folds = list(MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X, Y))
res, oof = run_cv(MODELS, X, Y, folds)
res.to_csv(f"{OUT}/05_text_model_results_raw.csv", index=False)

In [ ]:
# Results table in Lab 4 style: mean ± std over the 10 folds (percent)
fmt = lambda m, s: f"{m*100:.1f} ± {s*100:.1f}"
table = pd.DataFrame({
    "Model": res["Model"],
    "Macro F1 %": [fmt(m,s) for m,s in zip(res["macro_f1 mean"], res["macro_f1 std"])],
    "Precision %": [fmt(m,s) for m,s in zip(res["precision mean"], res["precision std"])],
    "Recall %": [fmt(m,s) for m,s in zip(res["recall mean"], res["recall std"])],
    "AUC %": [fmt(m,s) for m,s in zip(res["auc mean"], res["auc std"])],
    "Micro F1 %": [fmt(m,s) for m,s in zip(res["micro_f1 mean"], res["micro_f1 std"])],
    "Hamming loss": [f"{m:.3f}" for m in res["hamming mean"]],
    "Parameters": res["Parameters"],
})
table.to_csv(f"{OUT}/06_text_model_results_table.csv", index=False); display(table)

# Simplest model that is "good enough": within one std of the best macro F1
best = res.loc[res["macro_f1 mean"].idxmax()]
ok = res[res["macro_f1 mean"] >= best["macro_f1 mean"] - best["macro_f1 std"]]
simplest = ok.iloc[0]["Model"]   # MODELS is ordered simplest -> most complex
verdict = (f"Best macro F1: {best['Model']} ({best['macro_f1 mean']*100:.1f}%). "
           f"Simplest model within one std of it: {simplest}.")
print(verdict)

# chart
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(res["Model"], res["macro_f1 mean"]*100, yerr=res["macro_f1 std"]*100, capsize=6, color=["#9AA3BD","#FFC23D","#FF8A3D","#3CCB8A"])
ax.set_ylabel("Macro F1 (%)"); ax.set_title(f"Text labelling, {N_FOLDS}-fold stratified CV (mean ± std)")
for i, v in enumerate(res["macro_f1 mean"]*100): ax.text(i, v+1, f"{v:.1f}", ha="center")
plt.tight_layout(); plt.savefig(f"{OUT}/07_text_model_macro_f1.png", dpi=150); plt.show()

In [ ]:
# Per problem type: F1 from out-of-fold predictions (which types are hard?)
per = pd.DataFrame({m: f1_score(Y, oof[m], average=None, zero_division=0) for m in MODELS}, index=LABELS).mul(100).round(1)
per.insert(0, "share of complaints %", (Y.mean(0)*100).round(1))
per.to_csv(f"{OUT}/08_f1_per_problem_type.csv"); display(per)

## 6. Thai to English: machine translation of the text sample
We cannot read Thai, so we translate every complaint in the text sample with **NLLB-200** (Meta's open translation model).
- Needs a GPU: **Runtime → Change runtime type → T4 GPU**. About 15–30 minutes for 30,000 complaints.
- Progress is saved every 1,000 complaints in `translation_cache.csv`, so a re-run continues where it stopped. **Tip:** upload last run's `translation_cache.csv` into the `final_outputs` folder before running to skip complaints already translated.
- Without a GPU it translates only 1,500 complaints, so you can still test the steps.
- The models still learn from the **original Thai**. The English is for **us to read** and for the **Thai vs. English test** (RQ1).

In [ ]:
import torch
try:
    from transformers.utils import logging as hf_logging; hf_logging.set_verbosity_error()
except Exception: pass
lab = lab.copy(); lab["types_en"] = lab["types"].apply(lambda ts: ", ".join(en(t) for t in ts))
cache_path = f"{OUT}/translation_cache.csv"
cache = pd.read_csv(cache_path) if os.path.exists(cache_path) else pd.DataFrame(columns=["ticket_id","comment_en"])

def load_translator():
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(MT_MODEL, src_lang="tha_Thai")
    mt = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL, torch_dtype=torch.float16 if dev == "cuda" else torch.float32).to(dev).eval()
    eng = tok.convert_tokens_to_ids("eng_Latn")
    def translate(texts):
        out = []
        for i in range(0, len(texts), MT_BATCH):
            enc = tok(list(texts[i:i+MT_BATCH]), return_tensors="pt", padding=True, truncation=True, max_length=200).to(dev)
            with torch.no_grad():
                gen = mt.generate(**enc, forced_bos_token_id=eng, max_new_tokens=160, num_beams=1)
            out += tok.batch_decode(gen, skip_special_tokens=True)
        return out
    return translate, dev

if TRANSLATE:
    n_tr = len(lab) if torch.cuda.is_available() else min(1500, len(lab))
    if not torch.cuda.is_available():
        print("No GPU found. Translating only 1,500 complaints. For all of them: Runtime > Change runtime type > T4 GPU.")
    todo = lab.iloc[:n_tr]; todo = todo[~todo["ticket_id"].isin(set(cache["ticket_id"]))]
    if len(todo):
        translate, dev = load_translator()
        todo = todo.assign(_len=todo["comment"].str.len()).sort_values("_len")   # similar lengths batch faster
        t0 = time.time()
        for i in range(0, len(todo), 1000):
            part = todo.iloc[i:i+1000]
            cache = pd.concat([cache, pd.DataFrame({"ticket_id": part["ticket_id"].values,
                                                     "comment_en": translate(part["comment"].tolist())})])
            cache.to_csv(cache_path, index=False)
            print(f"translated {min(i+1000, len(todo)):,}/{len(todo):,}  ({time.time()-t0:.0f}s)")
lab_en = lab.merge(cache.drop_duplicates("ticket_id"), on="ticket_id", how="left")
print(f"English text ready for {lab_en['comment_en'].notna().sum():,} of {len(lab_en):,} complaints")
display(lab_en[["district_en","types_en","comment","comment_en"]].head(8))

In [ ]:
# Thai vs. English: same rows, same folds. English uses word TF-IDF (1-2 words); Thai uses character n-grams.
def tfidf_en(maxf): return TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2, max_features=maxf, sublinear_tf=True, stop_words="english")
MODELS_EN = {
  "Logistic Regression": (Pipeline([("tfidf", tfidf_en(50000)), ("clf", OneVsRestClassifier(LogisticRegression(C=4, solver="liblinear", class_weight="balanced", max_iter=1000)))]),
                          "English word TF-IDF 1-2, 50k; C=4, liblinear, balanced"),
  "Linear SVM":          (Pipeline([("tfidf", tfidf_en(50000)), ("clf", OneVsRestClassifier(LinearSVC(C=0.5, class_weight="balanced")))]),
                          "English word TF-IDF 1-2, 50k; C=0.5, balanced"),
}
mask = lab_en["comment_en"].notna().values
if mask.sum() >= 200:
    Xen, Xth, Ysub = lab_en.loc[mask, "comment_en"].fillna("").values, X[mask], Y[mask]
    if mask.all():
        folds_sub, res_th = folds, res[res["Model"].isin(MODELS_EN)]
    else:
        folds_sub = list(MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(Xth, Ysub))
        res_th, _ = run_cv({k: MODELS[k] for k in MODELS_EN}, Xth, Ysub, folds_sub, tag="[Thai]    ")
    res_en, oof_en = run_cv(MODELS_EN, Xen, Ysub, folds_sub, tag="[English] ")
    cmp = pd.DataFrame({"Model": res_en["Model"],
        "Thai macro F1 %": (res_th.set_index("Model").loc[res_en["Model"], "macro_f1 mean"].values*100).round(1),
        "English macro F1 %": (res_en["macro_f1 mean"].values*100).round(1),
        "English std %": (res_en["macro_f1 std"].values*100).round(1)})
    cmp["Difference (English - Thai)"] = (cmp["English macro F1 %"] - cmp["Thai macro F1 %"]).round(1)
    cmp.to_csv(f"{OUT}/09_thai_vs_english.csv", index=False); display(cmp)
    print("If the difference is within about one std, translation keeps enough meaning for labelling.")
else:
    cmp = None; print("Not enough translated complaints for the Thai vs. English test.")

In [ ]:
# What the model reads, in English: top words for each problem type (English Logistic Regression)
if cmp is not None:
    m = MODELS_EN["Logistic Regression"][0].fit(Xen, Ysub)
    vocab = np.array(m.named_steps["tfidf"].get_feature_names_out())
    kw = pd.DataFrame({LABELS[j]: vocab[np.argsort(est.coef_[0])[::-1][:10]]
                       for j, est in enumerate(m.named_steps["clf"].estimators_)})
    kw.index = [f"#{i+1}" for i in range(len(kw))]
    kw.to_csv(f"{OUT}/10_top_english_keywords_per_type.csv"); display(kw.T)

In [ ]:
# A file we can read: every translated complaint, the user's types, the model's types, and whether they disagree
if cmp is not None:
    pred = oof["Logistic Regression"][mask]          # Thai model, out-of-fold (honest) predictions
    read = lab_en.loc[mask, ["ticket_id","date","district_en","types_en","comment","comment_en"]].copy()
    read["model_types"] = [", ".join(l for l, v in zip(LABELS, row) if v) or "-" for row in pred]
    read["agree"] = [(set(a.split(", ")) & set(LABELS)) == set(b.split(", ")) - {"-"} for a, b in zip(read["types_en"], read["model_types"])]
    read = read.rename(columns={"comment": "comment_th", "district_en": "district", "types_en": "user_types"})
    read.sort_values("agree").to_csv(f"{OUT}/11_translated_complaints_to_check.csv", index=False, encoding="utf-8-sig")
    print(f"Model and user agree on {read['agree'].mean()*100:.1f}% of complaints. Disagreements are listed first in 11_translated_complaints_to_check.csv.")
    display(read[~read["agree"]].head(10)[["district","user_types","model_types","comment_en"]])

## 7. Where: first hotspot map (DBSCAN + heat map)
DBSCAN with the haversine distance groups complaints that are close together. `eps` = 150 m, `min_samples` = 25.
Shown for **Road** and **Flooding** complaints in 2024 – Oct 2025.

In [ ]:
import folium
from folium.plugins import HeatMap
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

EPS_M, MIN_S, R_EARTH = 150, 25, 6_371_000          # DBSCAN: 150 m radius, at least 25 complaints
ESRI_GRAY = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}"
ESRI_STREET = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}"
recent = df[(df["timestamp"] >= pd.Timestamp("2024-01-01", tz="Asia/Bangkok")) &
            (df["timestamp"] <= pd.Timestamp(TRAIN_END + " 23:59", tz="Asia/Bangkok"))]

fmap = folium.Map(location=[13.76, 100.54], zoom_start=11, tiles=None)   # Esri: free, no API key, works in Colab
folium.TileLayer(ESRI_GRAY, attr="Tiles &copy; Esri", name="Light gray map").add_to(fmap)
folium.TileLayer(ESRI_STREET, attr="Tiles &copy; Esri", name="Street map").add_to(fmap)

hot_rows = []
for thai, color in [("ถนน", "#d9480f"), ("น้ำท่วม", "#1c7ed6")]:      # Road = orange, Flooding = blue
    pts = recent[recent["types"].apply(lambda ts: thai in ts)]
    pts = pts.sample(min(20000, len(pts)), random_state=SEED)
    if len(pts) < MIN_S:
        print(f"{en(thai)}: not enough points"); continue
    xy = np.radians(pts[["lat", "lon"]].values)
    labels_ = DBSCAN(eps=EPS_M / R_EARTH, min_samples=MIN_S, metric="haversine", algorithm="ball_tree").fit_predict(xy)
    pts = pts.assign(cluster=labels_); core = labels_ >= 0
    sil = np.nan
    if core.sum() > 50 and len(set(labels_[core])) > 1:
        idx = np.random.RandomState(SEED).choice(np.where(core)[0], min(4000, core.sum()), replace=False)
        sil = silhouette_score(xy[idx], labels_[idx], metric="haversine")
    g = (pts[core].groupby("cluster")
         .agg(complaints=("ticket_id", "size"), lat=("lat", "mean"), lon=("lon", "mean"),
              district=("district_en", lambda s: s.mode().iat[0] if len(s.mode()) else "?"),
              median_fix_days=("fix_days", "median"))
         .sort_values("complaints", ascending=False))
    g.insert(0, "type", en(thai)); hot_rows.append(g.head(15))
    print(f"{en(thai)}: {len(pts):,} points, {g.shape[0]} hotspots, {(~core).mean()*100:.0f}% noise, silhouette {sil:.2f}")
    heat = folium.FeatureGroup(name=f"{en(thai)} heat map", show=(thai == "ถนน"))
    HeatMap(pts[["lat", "lon"]].values.tolist(), radius=8, blur=10, min_opacity=0.3,
            gradient={0.2: "#2b83ba", 0.5: "#abdda4", 0.75: "#fdae61", 1.0: "#d7191c"}).add_to(heat)
    heat.add_to(fmap)
    spots = folium.FeatureGroup(name=f"{en(thai)} top hotspots")
    for _, r in g.head(15).iterrows():
        fix = f"{r.median_fix_days:.1f} days" if pd.notna(r.median_fix_days) else "n/a"
        folium.CircleMarker([r.lat, r.lon], radius=4 + np.sqrt(r.complaints) / 2, color=color, fill=True,
            fill_opacity=0.75, weight=2,
            popup=folium.Popup(f"<b>{en(thai)} hotspot</b><br>{r.complaints} complaints<br>District: {r.district}<br>Median fix: {fix}", max_width=220)
        ).add_to(spots)
    spots.add_to(fmap)

legend = ('<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;padding:8px 12px;'
          'border-radius:6px;box-shadow:0 2px 8px rgba(0,0,0,.25);font:13px Arial"><b>Top hotspots</b><br>'
          '<span style="color:#d9480f">&#9679;</span> Road &nbsp;<span style="color:#1c7ed6">&#9679;</span> Flooding<br>'
          '<span style="color:#666">Bigger circle = more complaints</span></div>')
fmap.get_root().html.add_child(folium.Element(legend))
folium.LayerControl(collapsed=False).add_to(fmap)
fmap.save(f"{OUT}/12_hotspot_map.html")
hot = pd.concat(hot_rows) if hot_rows else pd.DataFrame()
hot.to_csv(f"{OUT}/13_top_hotspots.csv"); display(hot)
fmap

## 8. When: season, month, weekday and hour

In [ ]:
# complaints per month
monthly = df.set_index("timestamp").resample("MS").size()
fig, ax = plt.subplots(figsize=(10,3.6)); monthly.plot(ax=ax, color="#FF8A3D", lw=2)
ax.set_title("Complaints reported per month"); ax.set_ylabel("Complaints"); ax.set_xlabel("")
plt.tight_layout(); plt.savefig(f"{OUT}/14_complaints_per_month.png", dpi=150); plt.show()

# Seasonal index: average complaints per month in a season / average per month overall (1.0 = normal).
# Fair comparison, because the rainy season has 5 months and the hot season only 3.
per = df[df["labelled"] & (df["timestamp"] <= pd.Timestamp(TRAIN_END+" 23:59", tz="Asia/Bangkok"))].copy()
per["ym"] = per["timestamp"].dt.tz_localize(None).dt.to_period("M")
ex = per.explode("types").reset_index(drop=True); ex = ex[ex["types"].isin(top)]
monthly_t = ex.groupby(["types", "ym"]).size().rename("n").reset_index()
monthly_t["season"] = monthly_t["ym"].dt.month.map(SEASON)
idx_t = (monthly_t.groupby(["types", "season"])["n"].mean() / monthly_t.groupby("types")["n"].mean()).unstack()[["Hot","Rainy","Cool"]]
idx_t.index = idx_t.index.map(en)
all_m = per.groupby("ym").size().rename("n").reset_index(); all_m["season"] = all_m["ym"].dt.month.map(SEASON)
idx_t.loc["All complaints"] = (all_m.groupby("season")["n"].mean() / all_m["n"].mean())[["Hot","Rainy","Cool"]].values
season_idx = idx_t.round(2).sort_values("Rainy", ascending=False)
season_idx.to_csv(f"{OUT}/15_seasonal_index.csv"); display(season_idx)
print("Read it like this: 1.30 in Rainy = 30% more complaints per month than normal in the rainy season.")
ax = season_idx.plot(kind="barh", figsize=(9,5.5), color=["#FF8A3D","#1c7ed6","#9AA3BD"])
ax.axvline(1, color="black", lw=1, ls="--"); ax.set_xlabel("Seasonal index (1.0 = average month)")
ax.set_title("Which problems rise in each season (per-month average, fair across seasons)")
plt.tight_layout(); plt.savefig(f"{OUT}/16_seasonal_index.png", dpi=150); plt.show()

# hour x weekday heat map
hw = pd.crosstab(df["weekday"], df["hour"]); hw.index = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
fig, ax = plt.subplots(figsize=(11,3.6)); im = ax.imshow(hw.values, aspect="auto", cmap="magma")
ax.set_yticks(range(7)); ax.set_yticklabels(hw.index); ax.set_xticks(range(0,24,2)); ax.set_xlabel("Hour (Bangkok time)")
ax.set_title("When complaints are reported"); plt.colorbar(im, label="complaints")
plt.tight_layout(); plt.savefig(f"{OUT}/17_hour_weekday_heatmap.png", dpi=150); plt.show()
hw.to_csv(f"{OUT}/18_hour_weekday_counts.csv")

## 9. Why: daily rainfall vs. flooding complaints
Daily rainfall for central Bangkok from the free Open-Meteo historical weather API. This shows a **link**, not proof of cause.

In [ ]:
try:
    url = ("https://archive-api.open-meteo.com/v1/archive?latitude=13.75&longitude=100.50"
           f"&start_date={TRAIN_START}&end_date={TRAIN_END}&daily=precipitation_sum&timezone=Asia%2FBangkok")
    import requests
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=90); r.raise_for_status(); w = r.json()["daily"]; break
        except Exception as err:
            print(f"Weather download attempt {attempt+1} failed: {err}"); time.sleep(5)
    else:
        raise RuntimeError("weather API not reachable")
    rain = pd.DataFrame({"date": pd.to_datetime(pd.Series(w["time"])).dt.date, "rain_mm": w["precipitation_sum"]})
    flood = (df[df["types"].apply(lambda ts: "น้ำท่วม" in ts)].groupby("date").size().rename("flood_complaints").reset_index())
    daily = rain.merge(flood, on="date", how="left").fillna({"flood_complaints": 0})
    rho = daily[["rain_mm","flood_complaints"]].corr(method="spearman").iloc[0,1]
    daily.to_csv(f"{OUT}/19_daily_rain_vs_flooding.csv", index=False)
    print(f"Spearman correlation, daily rain vs. flooding complaints: {rho:.2f}")
    heavy = daily["rain_mm"] >= 20
    print(f"Flooding complaints per day: {daily.loc[heavy,'flood_complaints'].mean():.1f} on heavy-rain days (>= 20 mm) "
          f"vs {daily.loc[~heavy,'flood_complaints'].mean():.1f} on other days")
    fig, ax = plt.subplots(figsize=(10,3.6)); d = daily.set_index(pd.to_datetime(daily["date"]))
    d["flood_complaints"].rolling(7).mean().plot(ax=ax, color="#1c7ed6", label="Flooding complaints (7-day avg)")
    ax2 = ax.twinx(); d["rain_mm"].rolling(7).mean().plot(ax=ax2, color="#9AA3BD", label="Rain mm (7-day avg)")
    ax.set_title(f"Rain and flooding complaints (Spearman rho = {rho:.2f})"); ax.legend(loc="upper left"); ax2.legend(loc="upper right")
    plt.tight_layout(); plt.savefig(f"{OUT}/20_rain_vs_flooding.png", dpi=150); plt.show()
except Exception as e:
    rho = np.nan; print("Rainfall step skipped:", e)

## 10. Time-based test: does the text model work on months it never saw?
Random folds mix old and new complaints. Real use is different: we train on the past and label new complaints.
So we **train on June 2022 – Dec 2024** and **test on Jan – Oct 2025**. Same models, same settings.

In [ ]:
from sklearn.base import clone
t_split = pd.Timestamp(TEST_START, tz="Asia/Bangkok")
lab_all = df[df["labelled"] & (df["timestamp"] <= pd.Timestamp(TRAIN_END+" 23:59", tz="Asia/Bangkok")) & (df["comment"].str.len() >= 5)]
lab_all = lab_all[lab_all["types"].apply(lambda ts: any(t in top for t in ts))]
lab_all = (lab_all.assign(_norm=lab_all["comment"].str.lower().str.replace(r"\s+", "", regex=True))
                  .drop_duplicates("_norm").drop(columns="_norm"))
past, future = lab_all[lab_all["timestamp"] < t_split], lab_all[lab_all["timestamp"] >= t_split]
tr_df = past.sample(min(SAMPLE_N, len(past)), random_state=SEED)
te_df = future.sample(min(10000, len(future)), random_state=SEED)
to_Y = lambda d: np.array([[int(t in ts) for t in top] for ts in d["types"]])
Ytr, Yte = to_Y(tr_df), to_Y(te_df)
print(f"Train: {len(tr_df):,} complaints before {TEST_START}   Test: {len(te_df):,} complaints from {TEST_START} to {TRAIN_END}")

rows, per_time = [], {}
for name in ["Baseline (majority)", "Logistic Regression", "Linear SVM"]:
    m = clone(MODELS[name][0]).fit(tr_df["comment"].values, Ytr)
    yp, ys = m.predict(te_df["comment"].values), scores(m, te_df["comment"].values)
    try: a = roc_auc_score(Yte, ys, average="macro")
    except ValueError: a = np.nan
    cvm = res.set_index("Model").loc[name]
    rows.append({"Model": name,
                 "10-fold CV macro F1 %": round(cvm["macro_f1 mean"]*100, 1),
                 "2025 test macro F1 %": round(f1_score(Yte, yp, average="macro", zero_division=0)*100, 1),
                 "2025 precision %": round(precision_score(Yte, yp, average="macro", zero_division=0)*100, 1),
                 "2025 recall %": round(recall_score(Yte, yp, average="macro", zero_division=0)*100, 1),
                 "2025 AUC %": round(a*100, 1)})
    per_time[name] = f1_score(Yte, yp, average=None, zero_division=0)*100
time_test = pd.DataFrame(rows)
time_test["Drop (points)"] = (time_test["2025 test macro F1 %"] - time_test["10-fold CV macro F1 %"]).round(1)
time_test.to_csv(f"{OUT}/21_time_based_test.csv", index=False); display(time_test)
per_t = pd.DataFrame(per_time, index=LABELS).round(1)
per_t.to_csv(f"{OUT}/21b_time_test_f1_per_type.csv"); display(per_t)
print("A small drop means the model still works on new months. A big drop means complaints changed over time.")

## 11. ROC curves and the most confident mistakes
ROC curves use the **out-of-fold** scores of Linear SVM (each complaint scored by a model that never saw it).
Then we read the mistakes the model was most sure about, in English, to see **why** it fails.

In [ ]:
from sklearn.metrics import roc_curve
S = OOF_SCORES["Linear SVM"]; P_ = oof["Linear SVM"]
fig, ax = plt.subplots(figsize=(7, 6)); auc_rows = []
for j, l in enumerate(LABELS):
    if Y[:, j].min() == Y[:, j].max(): continue
    fpr, tpr, _ = roc_curve(Y[:, j], S[:, j]); a = roc_auc_score(Y[:, j], S[:, j])
    auc_rows.append({"type": l, "AUC %": round(a*100, 1), "share %": round(Y[:, j].mean()*100, 1)})
    ax.plot(fpr, tpr, lw=1.6, label=f"{l} ({a*100:.1f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Coin flip (50)")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("ROC curves, Linear SVM, out-of-fold (AUC %)"); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.savefig(f"{OUT}/22_roc_curves.png", dpi=150); plt.show()
auc_type = pd.DataFrame(auc_rows).sort_values("AUC %", ascending=False)
auc_type.to_csv(f"{OUT}/22b_auc_per_type.csv", index=False); display(auc_type)

# most confident mistakes: wrong label with the biggest score (false positive) or lowest score (missed type)
err = []
en_text = lab_en["comment_en"].values if "comment_en" in lab_en else np.array([np.nan]*len(lab))
for j, l in enumerate(LABELS):
    fp = np.where((P_[:, j] == 1) & (Y[:, j] == 0))[0]; fn = np.where((P_[:, j] == 0) & (Y[:, j] == 1))[0]
    for i in fp[np.argsort(-S[fp, j])][:5]:
        err.append({"type": l, "mistake": "said yes, user said no", "score": round(S[i, j], 2), "i": i})
    for i in fn[np.argsort(S[fn, j])][:5]:
        err.append({"type": l, "mistake": "missed it", "score": round(S[i, j], 2), "i": i})
err = pd.DataFrame(err)
if len(err):
    err["user types"] = [", ".join(en(t) for t in lab.iloc[i]["types"]) for i in err["i"]]
    err["district"] = [lab.iloc[i]["district_en"] for i in err["i"]]
    err["comment_th"] = [lab.iloc[i]["comment"] for i in err["i"]]
    err["comment_en"] = [en_text[i] for i in err["i"]]
    err = err.drop(columns="i")
    err.to_csv(f"{OUT}/23_confident_mistakes.csv", index=False, encoding="utf-8-sig")
    display(err.head(12)[["type", "mistake", "score", "user types", "comment_en"]])
print("Read 23_confident_mistakes.csv: many 'mistakes' are wrong user labels, not model errors (Ajarn's point).")

## 12. Fill in the empty problem types (Ajarn's feedback)
Many complaints have **no problem type** (100% from November 2025). We train the best text model (Linear SVM)
on **all** labelled complaints, then label the empty ones from their text. These predicted types feed the app.

In [ ]:
big = lab_all.sample(min(SAMPLE_N * 2, len(lab_all)), random_state=SEED)
final_text = clone(MODELS["Linear SVM"][0]).fit(big["comment"].values, to_Y(big))
unl = df[~df["labelled"] & (df["comment"].str.len() >= 5)]
pred_unl = np.vstack([final_text.predict(unl["comment"].values[i:i+50000]) for i in range(0, len(unl), 50000)]) if len(unl) else np.zeros((0, len(top)))
df["types_all"] = df["types"].apply(lambda ts: [t for t in ts if t in top])   # user types (top types only)
df["type_source"] = np.where(df["labelled"], "user", "none")
pred_lists = [[top[j] for j in np.where(r)[0]] for r in pred_unl]
df.loc[unl.index, "types_all"] = pd.Series(pred_lists, index=unl.index)
df.loc[unl.index, "type_source"] = "model"
fill = pd.DataFrame({"type": LABELS, "predicted complaints": pred_unl.sum(0).astype(int)})
fill["share of empty rows %"] = (fill["predicted complaints"] / max(len(unl), 1) * 100).round(1)
fill = fill.sort_values("predicted complaints", ascending=False)
no_type = int((pred_unl.sum(1) == 0).sum())
fill.to_csv(f"{OUT}/24_predicted_types_for_empty_rows.csv", index=False); display(fill)
print(f"Empty rows with text: {len(unl):,}. The model gave at least one type to {len(unl)-no_type:,} "
      f"({(1-no_type/max(len(unl),1))*100:.1f}%); {no_type:,} stay unknown.")
# save the text model for the website (Translator page runs it in the browser, no server)
vec, ovr = final_text.named_steps["tfidf"], final_text.named_steps["clf"]
inv = sorted(vec.vocabulary_.items(), key=lambda kv: kv[1])
web_model = {"labels": LABELS, "ngram": list(vec.ngram_range), "ngrams": [k for k, _ in inv],
             "idf": np.round(vec.idf_, 4).tolist(),
             "coef": [np.round(e.coef_.ravel(), 4).tolist() if hasattr(e, "coef_") else [0.0]*len(inv) for e in ovr.estimators_],
             "b": [round(float(e.intercept_[0]), 4) if hasattr(e, "intercept_") else -1.0 for e in ovr.estimators_],
             "trained_on": int(len(big)), "model": "TF-IDF char_wb 2-4 + Linear SVM (one-vs-rest)"}
with open(f"{OUT}/text_model.json", "w", encoding="utf-8") as f: json.dump(web_model, f, ensure_ascii=False, separators=(",", ":"))
print(f"text_model.json for the website: {os.path.getsize(f'{OUT}/text_model.json')/1e6:.1f} MB")
smp = unl.sample(min(300, len(unl)), random_state=SEED)
pd.DataFrame({"date": smp["date"].astype(str), "district": smp["district_en"],
              "predicted types": [", ".join(en(t) for t in df.at[i, "types_all"]) or "-" for i in smp.index],
              "comment_th": smp["comment"]}).to_csv(f"{OUT}/24b_predicted_sample.csv", index=False, encoding="utf-8-sig")

# ---------- save point: if Colab restarts after this, run cell 1 (setup) and the "Resume" cell below ----------
import pickle, gc
N_RAW, N_LAB = len(raw), len(lab)
KEEP = ["ticket_id", "timestamp", "date", "hour", "season", "district_en", "lat", "lon", "types", "types_all", "type_source",
        "labelled", "finished", "fix_days", "fix_outlier", "count_reopen"]
df = df[[c for c in KEEP if c in df.columns]].copy()          # drop the long text columns: later steps do not need them
recent = recent[[c for c in ["lat", "lon", "types", "district_en"] if c in recent.columns]].copy()
CK_NAMES = ["df", "recent", "hot", "LABELS", "top", "cmp", "daily", "mask", "read", "res", "rho", "season_idx", "time_test",
            "verdict", "DUP_REMOVED", "EPS_M", "MIN_S", "R_EARTH", "ESRI_GRAY", "N_RAW", "N_LAB"]
with open(f"{OUT}/checkpoint.pkl", "wb") as f:
    pickle.dump({k: globals()[k] for k in CK_NAMES if k in globals()}, f, protocol=pickle.HIGHEST_PROTOCOL)
for k in ["raw", "lab_all", "past", "future", "tr_df", "te_df", "big", "unl", "pred_unl", "pred_lists", "X", "OOF_SCORES", "oof", "lab_en"]:
    globals().pop(k, None)
gc.collect()
print(f"Save point written: {OUT}/checkpoint.pkl ({os.path.getsize(f'{OUT}/checkpoint.pkl')/1e6:.0f} MB). Memory freed.")

### Resume (only if Colab restarted)
If Colab restarted or disconnected after section 12 (error: `name 'df' is not defined`): run **cell 1 (setup)**, then **this cell**, then carry on from the section that failed. If nothing went wrong, this cell does nothing.

In [ ]:
if "df" not in globals():
    import pickle
    from sklearn.cluster import DBSCAN
    from sklearn.metrics import silhouette_score
    import folium
    ck_path = f"{OUT}/checkpoint.pkl"
    if not os.path.exists(ck_path):
        raise FileNotFoundError("No save point found. The runtime was reset, so run the notebook from the top (Runtime > Run all).")
    with open(ck_path, "rb") as f: globals().update(pickle.load(f))
    print(f"Resumed: {len(df):,} rows, {len(LABELS)} types. Continue with the next sections.")
else:
    print("Nothing to resume. Carry on.")

## 13. Where, part 2: are the hotspots real? (Getis-Ord Gi\*, HDBSCAN)
DBSCAN finds dense groups, but a busy area may be dense just because many people live there.
**Getis-Ord Gi\*** tests each 1 km grid cell: is it **significantly** higher than the city average, together with its neighbours?
z > 1.96 means hot at 95% confidence, z > 2.58 at 99%. We also compare DBSCAN with **HDBSCAN** (no fixed radius).

In [ ]:
from scipy.signal import convolve2d
CELL = 0.01                                  # about 1.1 km
gi_rows = []
gmap = folium.Map(location=[13.76, 100.54], zoom_start=11, tiles=None)
folium.TileLayer(ESRI_GRAY, attr="Tiles &copy; Esri", name="Light gray map").add_to(gmap)
lat0, lon0 = BKK_BOX["lat"][0], BKK_BOX["lon"][0]
nr = int((BKK_BOX["lat"][1]-lat0)/CELL)+1; nc = int((BKK_BOX["lon"][1]-lon0)/CELL)+1
gi_r = ((recent["lat"]-lat0)/CELL).astype(int).clip(0, nr-1); gi_c = ((recent["lon"]-lon0)/CELL).astype(int).clip(0, nc-1)
area = np.zeros((nr, nc), bool); area[gi_r, gi_c] = True       # study area = cells with any complaint
K3 = np.ones((3, 3))
for thai, color in [("ถนน", "#d9480f"), ("น้ำท่วม", "#1c7ed6")]:
    sel = recent["types"].apply(lambda ts: thai in ts).values
    grid = np.zeros((nr, nc)); np.add.at(grid, (gi_r[sel], gi_c[sel]), 1)
    x = grid[area]; n = area.sum(); xbar = x.mean(); s = np.sqrt((x**2).mean() - xbar**2)
    W = convolve2d(area.astype(float), K3, mode="same"); SX = convolve2d(grid*area, K3, mode="same")
    z = np.where(area, (SX - xbar*W) / (s*np.sqrt((n*W - W**2)/(n-1)) + 1e-9), np.nan)
    rr, cc = np.where(area & (z > 1.96))
    g = pd.DataFrame({"type": en(thai), "lat": lat0+(rr+.5)*CELL, "lon": lon0+(cc+.5)*CELL,
                      "complaints": grid[rr, cc].astype(int), "z": z[rr, cc].round(2)})
    g["confidence"] = np.where(g["z"] > 2.58, "99%", "95%")
    # district name of each hot cell = most common district of its complaints
    key = gi_r[sel].astype(str) + "_" + gi_c[sel].astype(str)
    dmode = recent[sel].groupby(key.values)["district_en"].agg(lambda s: s.mode().iat[0] if len(s.mode()) else "?")
    g["district"] = (pd.Series(rr).astype(str) + "_" + pd.Series(cc).astype(str)).map(dmode).values
    gi_rows.append(g.sort_values("z", ascending=False))
    # agreement with DBSCAN: share of DBSCAN hotspots that sit inside a Gi* hot cell
    hh = hot[hot["type"] == en(thai)] if len(hot) else hot
    if len(hh):
        hr = ((hh["lat"]-lat0)/CELL).astype(int); hc_ = ((hh["lon"]-lon0)/CELL).astype(int)
        inside_hot = (z[hr.clip(0, nr-1), hc_.clip(0, nc-1)] > 1.96).mean()*100
    else: inside_hot = np.nan
    print(f"{en(thai)}: {area.sum():,} cells, {len(g):,} hot cells (95%), {(g['z']>2.58).sum():,} at 99%; " +
          (f"{inside_hot:.0f}% of DBSCAN hotspots are inside a Gi* hot cell" if pd.notna(inside_hot) else "no DBSCAN hotspots to compare"))
    fg = folium.FeatureGroup(name=f"{en(thai)} Gi* hot cells", show=(thai == "ถนน"))
    for _, r in g.iterrows():
        folium.Rectangle([[r.lat-CELL/2, r.lon-CELL/2], [r.lat+CELL/2, r.lon+CELL/2]], color=color, weight=0.5,
                         fill=True, fill_opacity=0.65 if r.confidence == "99%" else 0.3,
                         popup=f"{en(thai)} · {r.district}<br>{r.complaints} complaints · z = {r.z} ({r.confidence})").add_to(fg)
    fg.add_to(gmap)
folium.LayerControl(collapsed=False).add_to(gmap); gmap.save(f"{OUT}/25_gistar_map.html")
gistar = pd.concat(gi_rows); gistar.to_csv(f"{OUT}/25b_gistar_hot_cells.csv", index=False); display(gistar.head(10))

# DBSCAN vs HDBSCAN on the same Road sample
try:
    from sklearn.cluster import HDBSCAN
    road_pts = recent[recent["types"].apply(lambda ts: "ถนน" in ts)]
    pts = road_pts.sample(min(10000, len(road_pts)), random_state=SEED)
    xy = np.radians(pts[["lat", "lon"]].values); cmp_rows = []
    for nm, mdl in [("DBSCAN (150 m, 25)", DBSCAN(eps=EPS_M/R_EARTH, min_samples=MIN_S, metric="haversine", algorithm="ball_tree")),
                    ("HDBSCAN (min size 25)", HDBSCAN(min_cluster_size=MIN_S, metric="haversine", algorithm="ball_tree"))]:
        lb = mdl.fit_predict(xy); core = lb >= 0; k = len(set(lb[core]))
        sil = np.nan
        if core.sum() > 50 and k > 1:
            ii = np.random.RandomState(SEED).choice(np.where(core)[0], min(3000, core.sum()), replace=False)
            sil = silhouette_score(xy[ii], lb[ii], metric="haversine")
        cmp_rows.append({"Method": nm, "clusters": k, "noise %": round((~core).mean()*100, 1), "silhouette": round(sil, 2)})
    clus_cmp = pd.DataFrame(cmp_rows); clus_cmp.to_csv(f"{OUT}/26_dbscan_vs_hdbscan.csv", index=False); display(clus_cmp)
except Exception as e:
    clus_cmp = None; print("HDBSCAN comparison skipped:", e)

## 14. Association rules: which places, seasons and hours go with which problems?
Each complaint becomes a "basket" of items: district, season, time of day, rain that day, problem types and fixing speed.
**FP-Growth** finds items that often appear together. We keep rules with **lift > 1.2** (the pair happens at least 20% more than by chance).

In [ ]:
from mlxtend.frequent_patterns import fpgrowth, association_rules
import inspect
BAND = lambda h: "Night 0-6" if h < 6 else "Morning 6-12" if h < 12 else "Afternoon 12-18" if h < 18 else "Evening 18-24"
ar = df[df["labelled"] & df["district_en"].notna() & (df["timestamp"] <= pd.Timestamp(TRAIN_END+" 23:59", tz="Asia/Bangkok"))]
ar = ar.sample(min(300000, len(ar)), random_state=SEED)
items = pd.DataFrame(index=ar.index)
for d in ar["district_en"].value_counts().index: items[f"district={d}"] = ar["district_en"].eq(d)
for s in ["Hot", "Rainy", "Cool"]: items[f"season={s}"] = ar["season"].eq(s)
for b in ["Night 0-6", "Morning 6-12", "Afternoon 12-18", "Evening 18-24"]: items[f"time={b}"] = ar["hour"].map(BAND).eq(b)
for t in top: items[f"type={en(t)}"] = ar["types"].apply(lambda ts, t=t: t in ts)
items["fix=slow (over 30 days)"] = ar["finished"] & ~ar["fix_outlier"] & ar["fix_days"].gt(30)
items["fix=fast (within 1 day)"] = ar["finished"] & ~ar["fix_outlier"] & ar["fix_days"].le(1)
if "daily" in globals():
    rmap = dict(zip(daily["date"], daily["rain_mm"])); rmm = ar["date"].map(rmap)
    items["rain=heavy (20 mm+)"] = rmm.ge(20).fillna(False); items["rain=dry (under 1 mm)"] = rmm.lt(1).fillna(False)
items = items.astype(bool)
freq = fpgrowth(items, min_support=0.002, use_colnames=True, max_len=3)
kw_ = {"num_itemsets": len(items)} if "num_itemsets" in inspect.signature(association_rules).parameters else {}
rules = association_rules(freq, metric="lift", min_threshold=1.2, **kw_)
is_out = lambda fs: len(fs) == 1 and next(iter(fs)).split("=")[0] in ("type", "fix")
rules = rules[rules["consequents"].apply(is_out)]
rules = rules[rules["confidence"] >= 0.2]
rules = rules[~rules.apply(lambda r: next(iter(r["consequents"])) in r["antecedents"], axis=1)]
rules["if"] = rules["antecedents"].apply(lambda s: " + ".join(sorted(s)))
rules["then"] = rules["consequents"].apply(lambda s: next(iter(s)))
rules = rules.sort_values(["lift", "confidence"], ascending=False)
rules_out = rules[["if", "then", "support", "confidence", "lift"]].round(3).head(60)
rules_out.to_csv(f"{OUT}/27_association_rules.csv", index=False)
print(f"{len(items):,} complaints, {items.shape[1]} items, {len(freq):,} frequent itemsets, {len(rules):,} rules kept")
display(rules_out.head(15))

## 15. Priority score: where should the city act first?
For each district and problem type in the **last 12 months of labelled data**:
- **How often** it is reported (complaints per month)
- **How slow** it is fixed (median fixing days, finished cases)
- **How often it is left unfinished** (share not finished)

Each part becomes a 0–1 rank across all district × type pairs. **Priority = 40% frequency + 40% slowness + 20% unfinished**, times 100.
The weights are **our choice**; the app shows each part so a reader can judge.

In [ ]:
end_ = pd.Timestamp(TRAIN_END+" 23:59", tz="Asia/Bangkok"); start_ = end_ - pd.DateOffset(months=12)
pr = df[df["labelled"] & df["district_en"].notna() & df["timestamp"].between(start_, end_)].explode("types")
pr = pr[pr["types"].isin(top)]
pr["fix_ok"] = pr["fix_days"].where(pr["finished"] & ~pr["fix_outlier"])
pri = (pr.groupby(["district_en", "types"])
         .agg(complaints=("ticket_id", "size"), median_fix_days=("fix_ok", "median"),
              unfinished=("finished", lambda s: 1 - s.mean()), reopened=("count_reopen", lambda s: (s > 0).mean()))
         .reset_index())
pri = pri[pri["complaints"] >= 30].copy()
pri["per month"] = (pri["complaints"] / 12).round(1)
pri["type"] = pri["types"].map(en); pri = pri.rename(columns={"district_en": "district"})
r_ = lambda s: s.rank(pct=True)
pri["priority"] = (100*(0.4*r_(pri["per month"]) + 0.4*r_(pri["median_fix_days"].fillna(pri["median_fix_days"].median()))
                        + 0.2*r_(pri["unfinished"]))).round(1)
pri["unfinished %"] = (pri["unfinished"]*100).round(1); pri["reopened %"] = (pri["reopened"]*100).round(1)
pri = pri.sort_values("priority", ascending=False)
pri_out = pri[["district", "type", "per month", "median_fix_days", "unfinished %", "reopened %", "priority"]].round(1)
pri_out.to_csv(f"{OUT}/28_priority_list.csv", index=False); display(pri_out.head(15))
dist_pri = (pri.assign(w=pri["complaints"]).groupby("district").apply(lambda g: np.average(g["priority"], weights=g["w"]))
               .round(1).sort_values(ascending=False).rename("district priority"))
dist_pri.to_csv(f"{OUT}/28b_district_priority.csv"); display(dist_pri.head(10).to_frame())

## 16. Forecast: how many complaints will each district get next week?
Weekly complaints per district (all types, 50 districts). Features: the last 4 weeks, 4- and 12-week averages, week of year and district.
**Train on weeks before 2025, test on 2025 onward** (future weeks only). Two simple baselines and two models:
- Naive: next week = last week
- 4-week average
- Linear Regression
- XGBoost

Metric: **MAE** (average error in complaints per district-week), reported as **mean ± std over the test weeks**.

In [ ]:
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
wk = df[df["district_en"].notna()].copy()
wk["week"] = wk["timestamp"].dt.tz_localize(None).dt.to_period("W-SUN").dt.start_time
last_full = wk["week"].max() - pd.Timedelta(days=7)            # drop the last (maybe half) week
panel = wk[wk["week"] <= last_full].groupby(["week", "district_en"]).size().unstack(fill_value=0).asfreq("W-MON", fill_value=0)
long = panel.stack().rename("y").reset_index().rename(columns={"district_en": "district"})
long = long.sort_values(["district", "week"])
g_ = long.groupby("district")["y"]
for k in [1, 2, 3, 4]: long[f"lag{k}"] = g_.shift(k)
long["avg4"] = g_.transform(lambda s: s.shift(1).rolling(4).mean())
long["avg12"] = g_.transform(lambda s: s.shift(1).rolling(12).mean())
long["weekofyear"] = long["week"].dt.isocalendar().week.astype(int); long["month"] = long["week"].dt.month
long["district_code"] = long["district"].astype("category").cat.codes
FEATS = ["lag1", "lag2", "lag3", "lag4", "avg4", "avg12", "weekofyear", "month", "district_code"]
data = long.dropna(subset=FEATS)
tr_, te_ = data[data["week"] < pd.Timestamp(TEST_START)], data[data["week"] >= pd.Timestamp(TEST_START)]
xgb = XGBRegressor(n_estimators=500, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                   objective="reg:squarederror", random_state=SEED, n_jobs=-1)
preds = {"Naive (last week)": te_["lag1"].values, "4-week average": te_["avg4"].values}
lr_f = LinearRegression().fit(tr_[FEATS], tr_["y"]); preds["Linear Regression"] = lr_f.predict(te_[FEATS]).clip(0)
xgb.fit(tr_[FEATS], tr_["y"]); preds["XGBoost"] = xgb.predict(te_[FEATS]).clip(0)
fc_rows = []
for nm, p in preds.items():
    e = te_.assign(ae=np.abs(te_["y"].values - p), se=(te_["y"].values - p)**2)
    per_w = e.groupby("week")[["ae", "se"]].mean()
    fc_rows.append({"Model": nm, "MAE mean": round(float(per_w["ae"].mean()), 2), "MAE std (over weeks)": round(float(per_w["ae"].std()), 2),
                    "RMSE": round(float(np.sqrt(e["se"].mean())), 2), "WAPE %": round(float(e["ae"].sum() / e["y"].sum() * 100), 1)})
fc = pd.DataFrame(fc_rows); fc.to_csv(f"{OUT}/29_forecast_results.csv", index=False); display(fc)
print(f"Train weeks: {tr_['week'].nunique()}, test weeks: {te_['week'].nunique()} (from {TEST_START} to {last_full.date()})")
imp = pd.Series(xgb.get_booster().get_score(importance_type="gain")).reindex(FEATS).fillna(0).sort_values()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
imp.plot(kind="barh", ax=ax[0], color="#FFC23D"); ax[0].set_title("XGBoost: which features matter (gain)")
tot = te_.assign(p=preds["XGBoost"]).groupby("week")[["y", "p"]].sum()
tot["y"].plot(ax=ax[1], color="#9AA3BD", label="Real"); tot["p"].plot(ax=ax[1], color="#FF8A3D", label="XGBoost forecast")
ax[1].set_title("All districts, weekly complaints (test period)"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/30_forecast.png", dpi=150); plt.show()

# next week, per district (for the app): features from the latest weeks
best_fc = fc.sort_values("MAE mean").iloc[0]["Model"]
nxt = long.groupby("district").tail(1).copy()
hist = {d: s.values for d, s in long.groupby("district")["y"]}
nrow = []
for _, r in nxt.iterrows():
    h = hist[r["district"]]; wnext = r["week"] + pd.Timedelta(days=7)
    nrow.append({"district": r["district"], "week": wnext, "lag1": h[-1], "lag2": h[-2], "lag3": h[-3], "lag4": h[-4],
                 "avg4": h[-4:].mean(), "avg12": h[-12:].mean(), "weekofyear": int(wnext.isocalendar()[1]),
                 "month": wnext.month, "district_code": r["district_code"]})
nxt = pd.DataFrame(nrow)
FC_FN = {"Naive (last week)": lambda X_: X_["lag1"].values, "4-week average": lambda X_: X_["avg4"].values,
         "Linear Regression": lambda X_: lr_f.predict(X_[FEATS]), "XGBoost": lambda X_: xgb.predict(X_[FEATS])}
nxt["forecast"] = np.clip(FC_FN[best_fc](nxt), 0, None).round(0).astype(int)      # use the model with the lowest MAE
nxt["last week"] = nxt["lag1"].astype(int)
nxt[["district", "week", "last week", "forecast"]].to_csv(f"{OUT}/31_next_week_forecast.csv", index=False)
print(f"Best model by MAE: {best_fc}. Next week starts {nxt['week'].iloc[0].date()}"); display(nxt.sort_values("forecast", ascending=False).head(10)[["district", "last week", "forecast"]])

## 17. Data for the monitoring app
One small JSON file (`app_data.json`) feeds the app on our project page: district map, filters by type, season and hour,
priority list, rules, next-week forecast and a reader of translated complaints. Empty types use the model's labels from section 12.

In [ ]:
def jnum(x, nd=2):
    try: return None if pd.isna(x) else round(float(x), nd)
    except Exception: return None
dd = df[df["district_en"].notna()].copy()
dd["band"] = dd["hour"].map(BAND)
cent = dd.groupby("district_en")[["lat", "lon"]].median()
ex_all = dd[["district_en", "types_all", "season", "band", "type_source"]].explode("types_all").dropna(subset=["types_all"])
ex_all["type"] = ex_all["types_all"].map(en)
cube = ex_all.groupby(["district_en", "type", "season", "band"]).size()
SEAS, BANDS = ["Hot", "Rainy", "Cool"], ["Night 0-6", "Morning 6-12", "Afternoon 12-18", "Evening 18-24"]
fixd = dd[dd["finished"] & ~dd["fix_outlier"]].groupby("district_en")["fix_days"].median()
unf = 1 - dd.groupby("district_en")["finished"].mean()
fcd = nxt.set_index("district") if "nxt" in globals() else pd.DataFrame()
districts = []
for d, (la, lo) in cent.iterrows():
    c = {t: [[int(cube.get((d, t, s, b), 0)) for b in BANDS] for s in SEAS] for t in LABELS}
    districts.append({"name": d, "lat": jnum(la, 5), "lon": jnum(lo, 5), "cube": c,
                      "total": int((dd["district_en"] == d).sum()), "medianFix": jnum(fixd.get(d)), "unfinished": jnum(unf.get(d)*100, 1),
                      "priority": jnum(dist_pri.get(d), 1) if "dist_pri" in globals() else None,
                      "lastWeek": int(fcd.loc[d, "last week"]) if d in fcd.index else None,
                      "forecast": int(fcd.loc[d, "forecast"]) if d in fcd.index else None})
reader = []
if "read" in globals():
    rs = read.sample(min(400, len(read)), random_state=SEED)
    reader = [{"d": r["district"] if isinstance(r["district"], str) else "?", "u": r["user_types"], "m": r["model_types"],
               "t": r["comment_en"], "a": bool(r["agree"])} for _, r in rs.iterrows()]
app = {
    "updated": pd.Timestamp.now(tz="Asia/Bangkok").strftime("%Y-%m-%d"),
    "dataStart": str(df["timestamp"].min().date()), "dataEnd": str(df["timestamp"].max().date()),
    "rows": int(len(df)), "modelFilled": int((df["type_source"] == "model").sum()),
    "types": LABELS, "seasons": SEAS, "bands": BANDS, "districts": districts,
    "priority": pri_out.head(60).to_dict("records") if "pri_out" in globals() else [],
    "rules": rules_out.head(40).to_dict("records") if "rules_out" in globals() else [],
    "hotspots": hot.reset_index(drop=True)[["type", "complaints", "lat", "lon", "district", "median_fix_days"]].round(5).to_dict("records") if len(hot) else [],
    "gistar": gistar.head(400).to_dict("records") if "gistar" in globals() else [],
    "forecast": {"model": best_fc, "week": str(nxt["week"].iloc[0].date()), "table": fc.to_dict("records")} if "nxt" in globals() else None,
    "reader": reader,
}
with open(f"{OUT}/app_data.json", "w", encoding="utf-8") as f: json.dump(app, f, ensure_ascii=False, default=str)
print(f"app_data.json: {os.path.getsize(f'{OUT}/app_data.json')/1024:.0f} KB, {len(districts)} districts, {len(reader)} translated complaints")

## 18. Final summary, and download everything

In [ ]:
L = []
L.append(f"Final summary (auto-generated {pd.Timestamp.now(tz='Asia/Bangkok'):%Y-%m-%d %H:%M})")
L.append(f"- Data: {N_RAW:,} raw rows -> {len(df):,} clean rows ({df['timestamp'].min().date()} to {df['timestamp'].max().date()}). See 02_cleaning_log.csv.")
L.append(f"- Empty problem type: {(~df['labelled']).mean()*100:.1f}% of clean rows; text model filled {(df['type_source']=='model').sum():,} of them.")
L.append(f"- Text set: {N_LAB:,} complaints, {len(LABELS)} types ({', '.join(LABELS)}), {N_FOLDS}-fold stratified CV, seed {SEED}; {DUP_REMOVED:,} duplicates removed.")
for _, r in res.iterrows():
    L.append(f"  - {r['Model']}: macro F1 {r['macro_f1 mean']*100:.1f} ± {r['macro_f1 std']*100:.1f}%, precision {r['precision mean']*100:.1f}%, recall {r['recall mean']*100:.1f}%, AUC {r['auc mean']*100:.1f}%")
L.append(f"- {verdict}")
L.append("- Time-based test (train < 2025, test 2025): " + "; ".join(f"{r['Model']} {r['2025 test macro F1 %']}% (CV {r['10-fold CV macro F1 %']}%)" for _, r in time_test.iterrows()))
if cmp is not None:
    L.append("- Thai vs. English: " + "; ".join(f"{r['Model']} {r['Thai macro F1 %']}% vs {r['English macro F1 %']}% (n={int(mask.sum()):,})" for _, r in cmp.iterrows()))
L.append(f"- Hotspots: DBSCAN {len(hot)} listed; Gi* hot cells: " + ", ".join(f"{t} {n}" for t, n in gistar['type'].value_counts().items()))
if clus_cmp is not None: L.append("- DBSCAN vs HDBSCAN: " + "; ".join(f"{r['Method']}: {r['clusters']} clusters, silhouette {r['silhouette']}" for _, r in clus_cmp.iterrows()))
L.append(f"- Seasonal index (Rainy): " + ", ".join(f"{k} {v:.2f}" for k, v in season_idx["Rainy"].head(4).items()))
L.append(f"- Rain vs flooding complaints (Spearman): {rho:.2f}")
L.append("- Top rules: " + " | ".join(f"{r['if']} -> {r['then']} (lift {r['lift']})" for _, r in rules_out.head(3).iterrows()))
L.append("- Top priority: " + " | ".join(f"{r['district']} {r['type']} ({r['priority']})" for _, r in pri_out.head(3).iterrows()))
L.append("- Forecast (MAE per district-week): " + "; ".join(f"{r['Model']} {r['MAE mean']} ± {r['MAE std (over weeks)']}" for _, r in fc.iterrows()))
summary = "\n".join(L); open(f"{OUT}/00_summary.txt", "w").write(summary); print(summary)

ZIP = "final_outputs.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT)):
        if f != "checkpoint.pkl": z.write(os.path.join(OUT, f), f)     # the save point is big and only for Colab
print(f"{ZIP}: {os.path.getsize(ZIP)/1e6:.1f} MB")
try:
    from google.colab import files; files.download(ZIP)
except Exception:
    print("Saved", ZIP)